In [21]:
import numpy as np 
from time import perf_counter
from numba import njit

In [22]:
# L2 cache - 16Mb
N = 17_000_000
x = np.ones(N, dtype=np.float64)

@njit
def soma_stride(x, stride):
    s = 0.0
    for i in range(0, x.size, stride):
        s += x[i]
    return s

In [24]:
for stride in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    t0 = perf_counter() 
    s = soma_stride(x, stride)
    dt = perf_counter() - t0 
    acessos = (N + stride - 1) // stride
    print(stride, dt, dt/acessos * 1e9, 'ns/acesso', s)

1 0.018541657998866867 1.0906857646392274 ns/acesso 17000000.0
2 0.008870254998328164 1.0435594115680193 ns/acesso 8500000.0
4 0.004754290996061172 1.1186567049555698 ns/acesso 4250000.0
8 0.0039023019999149255 1.8363774117246707 ns/acesso 2125000.0
16 0.0034967179963132367 3.2910287024124583 ns/acesso 1062500.0
32 0.0017198290006490424 3.237325177692315 ns/acesso 531250.0
64 0.0011982529977103695 4.511070109027274 ns/acesso 265625.0
128 0.0006608069961657748 4.975469239952225 ns/acesso 132813.0
256 0.00034451400279067457 5.187916978491342 ns/acesso 66407.0


In [11]:
# L2 cache - 16Mb
N = 3000
A = np.random.random((N,N))

def soma_linhas(A):
    s = 0.0 
    n, m = A.shape
    for i in range(n):
        for j in range(m):
            s += A[i,j]

    return s

def soma_colunas(A):
    s = 0.0
    n, m = A.shape 
    for j in range(m):
        for i in range(n):
            s += A[i, j]

    return s

In [13]:
for f in (soma_linhas, soma_colunas):
    t0 = perf_counter()
    r = f(A)
    print(f.__name__, perf_counter() - t0, r)

soma_linhas 0.9188048030046048 4500377.431844127
soma_colunas 0.6979707380014588 4500377.4318437455


In [15]:
for mib in [0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128, 256]:
    n = int(mib * 1024**2 / 8)
    x = np.ones(n, dtype=np.float64)

    _ = x.sum()

    tempos = []
    for _ in range(7):
        t0 = perf_counter()
        r = x.sum()
        tempos.append(perf_counter() - t0)

    med = np.median(tempos)
    print(f'{mib:7.2f} Mib {med*1e3:8.3f} ms {med/n*1e9:8.3f} ns/elem')

   0.25 Mib    0.017 ms    0.521 ns/elem
   0.50 Mib    0.014 ms    0.212 ns/elem
   1.00 Mib    0.028 ms    0.216 ns/elem
   2.00 Mib    0.052 ms    0.199 ns/elem
   4.00 Mib    0.107 ms    0.204 ns/elem
   8.00 Mib    0.203 ms    0.194 ns/elem
  16.00 Mib    0.587 ms    0.280 ns/elem
  32.00 Mib    0.895 ms    0.213 ns/elem
  64.00 Mib    2.124 ms    0.253 ns/elem
 128.00 Mib    3.983 ms    0.237 ns/elem
 256.00 Mib    7.458 ms    0.222 ns/elem


In [16]:
import tracemalloc

def medir(func, *args):
    tracemalloc.start()
    t0 = perf_counter()
    result = func(*args)
    dt = perf_counter() - t0
    atual, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, dt, pico / 1024**2
# resultado, tempo, pico_mib = medir(minha_funcao, entrada)

In [25]:
@njit
def soma_linhas_jit(A):
    s = 0.0
    n, m = A.shape
    for i in range(n):
        for j in range(m):
            s += A[i, j]
    return s

@njit
def soma_colunas_jit(A):
    s = 0.0
    n, m = A.shape
    for j in range(m):
        for i in range(n):
            s += A[i, j]
    return s

In [26]:
for f in (soma_linhas_jit, soma_colunas_jit):
    t0 = perf_counter()
    r = f(A)
    print(f.__name__, perf_counter() - t0, r)

soma_linhas_jit 0.07857184499880532 4500377.431844127
soma_colunas_jit 0.08040824400086422 4500377.4318437455
